# StarLayer User Guide (Notebook)

This notebook is located at https://github.com/hidden-graph/starlayer/blob/main/docs/user-guide-v1.ipynb

## How to run this notebook

1. pip install from the github repository.
2. Run cells from top to bottom so shared variables remain available.


In [2]:
pip install "git+https://github.com/hidden-graph/starlayer.git"

  Cloning https://github.com/hidden-graph/starlayer.git to /tmp/pip-req-build-qd9sapem
  Running command git clone --filter=blob:none --quiet https://github.com/hidden-graph/starlayer.git /tmp/pip-req-build-qd9sapem
  Resolved https://github.com/hidden-graph/starlayer.git to commit 14c985ff90db5e41a6dc6091747b4ea4b0abec6c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 7.5 MB/s eta 0:00:00
  Created wheel for starlayer: filename=starlayer-0.1.0-py3-none-any.whl size=389140 sha256=e61ec0d0a23aea57a1ca1e89937dd12236424e7d28ff8d160f63ba661185dce4
  Stored in directory: /tmp/pip-ephem-wheel-cache-ejeahc_y/wheels/2b

In [3]:
from starlayergraph import StarLayerGraph, Namespace, TripleTerm, DirLangString, Literal
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## 1. Graph and literal semantics
Extension of the rdflib graph model to support RDF 1.2.  
- Triple terms and statement resources
- Reification via rdf:reifies and statement metadata
- Direction-tagged strings such as "hello"@en--ltr and "مرحبا"@ar--rtl

In [4]:
#create the graph, and assign namespace
g = StarLayerGraph()
g.bind("ex", EX)

#create a triple term
tt = TripleTerm(EX.bob, EX.knows, EX.carol)

#create a reifer (EX.clain) associated with triple term and add to graph
g.add_reification(EX.claim, tt)
g.add((EX.claim, EX.source, EX.wikipedia))

print((EX.claim, RDF.reifies, tt) in g)
print(g.serialize(format="turtle12"))

True
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim ex:source ex:wikipedia ;
    rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



In [19]:
g = StarLayerGraph()
g.bind("ex", EX)

#add reifiers to the graph
g.add((EX.claim, RDF.reifies, (EX.bob, EX.knows, EX.carol)))
g.add((EX.other, RDF.reifies, (EX.bob, EX.likes, EX.dana)))

#triples accepts triple term as object to select triples
selectTriples = g.triples((None, None, (EX.bob, EX.knows, EX.carol)))

for s, p, o in selectTriples:
    print(g.qname(s), g.qname(p), o)
for t in g.triple_terms(subject=EX.bob):
    print(t)
print(g.has_triple_term(EX.bob, EX.knows, EX.carol))
print(g.has_triple_term(EX.bob, EX.knows, EX.dana))

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:likes ex:dana )>>
True
False


In [40]:
#RDF.reifies is the common approach to making a statement about a statement, RDF 1.2 allows triple terms in object position of any triple.



#add reifiers to the graph
g.add((EX.dana, EX.said, (EX.bob, EX.knows, EX.carol)))


#triples accepts triple term as object to select triples
selectTriples = g.triples((None, None, (EX.bob, EX.knows, EX.carol)))

#qname_term is a starlayer function that adds qname transformation to triple terms as well.
for s, p, o in selectTriples:
    print(g.qname_term(s), g.qname_term(p), g.qname_term(o))
#NOTE - revise to use new qname(term) on s,p and o.


AttributeError: 'StarLayerGraph' object has no attribute 'qname_term'

### Finding what's been said about a statement
`reifiers()`, `reifications()`, `reifier_annotations()`, `reified_triples()`, and `remove_reification()` navigate the reifier ↔ triple-term ↔ annotation relationship directly, without writing SPARQL.

In [22]:
# continues using g from the previous cell
g.add((EX.claim, EX.source, EX.wikipedia))

tt1 = (EX.bob, EX.knows, EX.carol)

# reifiers(): which reifier node(s) reify a given triple term?
print([g.qname(r) for r in g.reifiers(TT=tt1)])

# reifications(): which triple terms have at least one reifier?
for tt in g.reifications():
    print(tt)

# reifier_annotations(): a reifier's own annotation triples (excludes rdf:reifies itself)
for reifier, pred, val in g.reifier_annotations(tt1):
    print(g.qname(reifier), g.qname(pred), g.qname(val))

# reified_triples(): the triple term(s) a specific reifier reifies
for tt in g.reified_triples(EX.claim):
    print(tt)

# remove_reification(): undo just the rdf:reifies triple; annotations survive
g.remove_reification(EX.claim)
print((EX.claim, RDF.reifies, tt1) in g)
print((EX.claim, EX.source, EX.wikipedia) in g)

['ex:claim']
<<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:likes ex:dana )>>
ex:claim ex:source ex:wikipedia
<<( ex:bob ex:knows ex:carol )>>
False
True


### Set the direction of a string literal.

DirLangString has been added to set the direction of a string literal.


In [27]:
g = StarLayerGraph()
g.bind("ex", EX)

#literals can include language direction.
g.add((EX.title, EX.value, DirLangString("مرحبا", "ar", "rtl")))
g.add((EX.title, EX.value, Literal("hello","en")))
g.add((EX.title, EX.value, DirLangString("hello","en","ltr")))

print(g.serialize(format="turtle12"))

@version "1.2" .
@prefix ex: <http://example.org/> .

ex:title ex:value "hello"@en, "مرحبا"@ar--rtl, "hello"@en--ltr .



## 2. Parsing and Serialization
Supports parsing and serialization of RDF 1.2 content across the full set of rdflib supported formats. (turtle12 and longturtle12 for Turtle, nt12 and nq12 for N-Triples and N-Quads, trig12 and trix12 for datasets, rdfxml12,  and jsonld12. (jsonld does not have a published RDF 1.2 spec.)
- Quoted triple-term content
- Turtle annotation syntax
- Language-direction literals

In [37]:
g_parsed = StarLayerGraph()
g_parsed.bind("ex", EX)
#NOTE - I think we are missing a common annotation form.  {(s,p,o)} ex:says ex:bob
# parse mixed RDF 1.2 content: direction-tagged literals, quoted triples, and multiple reification forms
g_parsed.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

    # language-direction literals
    ex:note_en ex:text "hello"@en--ltr .
    ex:note_ar ex:text "مرحبا"@ar--rtl .

    # canonical reification with rdf:reifies
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> ;
      ex:source ex:wikipedia ;
      ex:confidence "high" .

    # anonymous inline annotation block
    ex:bob ex:likes ex:dana {| ex:since "2020" ; ex:source ex:LinkedIn |} .

    # named reifier with annotations
    ex:bob ex:mentions ex:erin ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:WikiData |} .

    # named reifier without annotation block
    ex:bob ex:worksWith ex:frank ~ ex:stmt2 .

    # an additional  quoted triple term reused in querie examples.
    ex:alice ex:mentions <<( ex:bob ex:likes ex:dana )>> .
''', format='turtle12')

print(g_parsed.serialize(format='turtle12'))
#other options:
#turtle12, longturtle12, nt12, nq12, trig12, trix12, rdfxml12,jsonld12


@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:alice ex:mentions <<( ex:bob ex:likes ex:dana )>> .

ex:bob ex:likes ex:dana {| ex:since "2020" ; ex:source ex:LinkedIn |} ;
    ex:mentions ex:erin ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:WikiData |} ;
    ex:worksWith ex:frank ~ ex:stmt2 .

ex:claim ex:confidence "high" ;
    ex:source ex:wikipedia ;
    rdf:reifies <<( ex:bob ex:knows ex:carol )>> .

ex:note_ar ex:text "مرحبا"@ar--rtl .

ex:note_en ex:text "hello"@en--ltr .



## 3. SPARQL and query semantics
Supports SPARQL 1.2 query execution.
- Query over reified quoted triples
- Uses Turtle 1.2 quoted-triple syntax (`<<( ... )>>`)
- Binds variables for terms inside triple terms (`?s ?p ?o`)

Following examples use the graph parsed in section 2 above.

In [38]:
# bind all terms inside a quoted triple and include statement metadata
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?s ?p ?o ?source WHERE {
  ?claim rdf:reifies <<( ?s ?p ?o )>> .
  ?claim ex:source ?source .
  FILTER(?p = ex:knows)
}
ORDER BY ?claim ?s ?o
""")

for row in rows:
    print(
        g_parsed.qname(row.claim),
        g_parsed.qname(row.s),
        g_parsed.qname(row.p),
        g_parsed.qname(row.o),
        g_parsed.qname(row.source),
    )

ex:claim ex:bob ex:knows ex:carol ex:wikipedia


In [ ]:
# selective term binding inside a quoted triple pattern
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?person ?friend WHERE {
  ?claim rdf:reifies <<( ?person ex:knows ?friend )>> .
  FILTER(?friend = ex:carol)
}
ORDER BY ?claim ?person
""")

for row in rows:
    print(
        g_parsed.qname(row.claim),
        g_parsed.qname(row.person),
        g_parsed.qname(row.friend),
    )

ex:claim ex:bob ex:carol


In [ ]:
# detect triple-term values dynamically with isTRIPLE
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim WHERE {
  ?claim rdf:reifies ?statement .
  FILTER( isTRIPLE(?statement) )
}
ORDER BY ?claim
""")

for row in rows:
    print(g_parsed.qname(row.claim))

ex:claim
ex:stmt1
ex:stmt2
rr:0


### Additional SPARQL 1.2 functions
`TRIPLE(s, p, o)` is the function-call spelling of `<<( s p o )>>`. `SUBJECT()`/`PREDICATE()`/`OBJECT()` pull the three components back out of a bound triple term. `LANGDIR()`/`hasLANGDIR()`/`STRLANGDIR()` work with base direction directly; `LANG()`/`hasLANG()` are the ordinary rdflib functions, extended to also recognize a direction-tagged literal.

In [41]:
# TRIPLE() and SUBJECT()/PREDICATE()/OBJECT()
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?s ?p ?o WHERE {
  ex:claim rdf:reifies ?t .
  FILTER(?t = TRIPLE(ex:bob, ex:knows, ex:carol))
  BIND(SUBJECT(?t) AS ?s)
  BIND(PREDICATE(?t) AS ?p)
  BIND(OBJECT(?t) AS ?o)
}
""")
for row in rows:
    print(g_parsed.qname(row.s), g_parsed.qname(row.p), g_parsed.qname(row.o))

ex:bob ex:knows ex:carol


In [42]:
# LANGDIR() / hasLANGDIR() / LANG() / hasLANG() over the direction-tagged notes
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?s ?lang ?dir ?hasDir WHERE {
  ?s ex:text ?lit .
  BIND(LANG(?lit) AS ?lang)
  BIND(LANGDIR(?lit) AS ?dir)
  BIND(hasLANGDIR(?lit) AS ?hasDir)
}
ORDER BY ?s
""")
for row in rows:
    print(g_parsed.qname(row.s), row.lang, row.dir, row.hasDir)

# STRLANGDIR() constructs a direction-tagged literal directly from plain strings
rows = g_parsed.query('SELECT ?lit WHERE { BIND(STRLANGDIR("hi", "en", "ltr") AS ?lit) }')
for row in rows:
    print(row.lit.n3())

ex:note_ar ar rtl true
ex:note_en en ltr true
"hi"@en--ltr


### TTL annotation shorthand inside SPARQL queries
The `{| ?pred ?val |}`, `~ ?r`, and `<< s p o >>` forms used to *parse* `g_parsed` above (see the Turtle 1.2 content in the parsing section) also work directly inside a SPARQL WHERE clause — you query the shorthand the same way you wrote it, without expanding to `rdf:reifies`/`<<( )>>` by hand.

In [43]:
# {| ?pred ?val |}: query an anonymous reifier's annotations inline
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pred ?val WHERE {
  ex:bob ex:likes ex:dana {| ?pred ?val |}
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname(row.pred), row.val)

# ~ ?r: bind the reifier itself for a named reifier
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?r WHERE {
  ex:bob ex:worksWith ex:frank ~ ?r
}
""")
for row in rows:
    print(g_parsed.qname(row.r))

# << s p o >> ?pred ?val: reification shorthand, no assertion required
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pred ?val WHERE {
  << ex:bob ex:likes ex:dana >> ?pred ?val
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname(row.pred), row.val)

ex:since 2020
ex:source http://example.org/LinkedIn
ex:stmt2
ex:since 2020
ex:source http://example.org/LinkedIn


## 4. SHACL validation and rules

- Validation over RDF 1.2 graphs
- SHACL rule support
- Direction-aware datatype constraints

In [ ]:
#NOTE - this does not show any incremental SHACL functionality.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ;
      ex:age 30 .
""", format="turtle")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:age ; sh:minCount 1 ; sh:datatype xsd:integer ] .
""", format="turtle")
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes)
print(result.conforms)

True


In [44]:
#NOTE -
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
""", format="turtle")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonRule a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:rule [ a sh:TripleRule ; sh:subject sh:this ; sh:predicate ex:inferred ; sh:object ex:yes ] .
""", format="turtle")
result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print((EX.alice, EX.inferred, EX.yes) in result.data_graph)
print(result.conforms)

True
True


## 5. Backend graph-store and format support

- In-memory backend and native RDF 1.2 backend modes
- Eight RDF 1.2-aware formats: turtle12, nt12, nq12, trig12, trix12, rdfxml12, jsonld12, longturtle12
- Dual-mode RDF 1.1 and RDF 1.2 operation

In [ ]:
g = StarLayerGraph()
g.bind("ex", EX)
g.add((EX.alice, EX.claims, (EX.bob, EX.knows, EX.carol)))
print(g.serialize(format="turtle12"))
print("---")
print(g.serialize(format="nt12"))

### Switching to the native backend

```python
g = StarLayerGraph()
g = StarLayerGraph(backend='rdf-1.2')
```